# Walkthrough: reserving one segment end to end

This notebook is a readable tour of the project. All logic lives in `src/`; the notebook only calls it.
Segment: **commercial auto liability (Erie Ins Exchange Grp)**, CAS Schedule P, valuation date 31.12.2007.
Amounts in USD thousands. The full analysis of all three segments is `python -m src.pipeline`.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd

from src.data import load_segments, upper_triangle
from src.deterministic import chain_ladder, volume_weighted_ldf
from src.mack import mack
from src.merz_wuthrich import merz_wuthrich
from src.bootstrap import odp_bootstrap, odp_one_year_cdr
from src.priors import loss_ratio_trend, trended_mature_prior, generalised_cape_cod
from src.deterministic import bornhuetter_ferguson, apriori_elr_from_mature_years
from src.tail_uncertainty import paid_tail_candidates, lognormal_tail_sigma

pd.set_option("display.float_format", lambda x: f"{x:,.0f}" if abs(x) >= 100 else f"{x:.4g}")
seg = load_segments()["comauto"]
paid, inc = upper_triangle(seg.paid), upper_triangle(seg.incurred)
pd.DataFrame(paid, index=seg.origins, columns=seg.lags).round(0)

,12,24,36,48,60,72,84,96,108,120
1998,"23,185","43,679","54,850","63,761","69,765","72,702","74,702","75,658","75,908","76,004"
1999,"24,498","41,011","52,781","64,413","72,172","75,408","77,862","78,573","78,969",NaN
2000,"27,791","50,111","66,713","79,626","88,941","96,576","97,805","98,628",NaN,NaN
2001,"31,568","56,519","75,597","94,990","109,225","116,764","119,531",NaN,NaN,NaN
2002,"35,169","60,541","88,380","104,609","115,389","122,439",NaN,NaN,NaN,NaN
2003,"39,917","76,979","104,357","132,191","144,132",NaN,NaN,NaN,NaN,NaN
2004,"39,022","75,840","105,570","120,748",NaN,NaN,NaN,NaN,NaN,NaN
2005,"38,757","68,071","92,370",NaN,NaN,NaN,NaN,NaN,NaN,NaN
2006,"35,714","65,467",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2007,"38,963",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 1. Chain ladder and Mack (1993)
Volume-weighted factors; Mack standard error by accident year (no tail inside the triangle).

In [2]:
cl_p = chain_ladder(paid)
m = mack(paid)
print("LDFs:", np.round(volume_weighted_ldf(paid), 4))
pd.DataFrame({"latest": m.latest, "reserve": m.reserve.round(0), "mack_se": m.se.round(0), "cv_%": (100 * m.cv).round(1)},
             index=seg.origins)

LDFs: [1.8206 1.3551 1.2045 1.1113 1.0623 1.0234 1.0099 1.0042 1.0013]


,latest,reserve,mack_se,cv_%
1998,"76,004",0,0,NaN
1999,"78,969",100,70,70
2000,"98,628",538,160,29.8
2001,"119,531","1,848",337,18.3
2002,"122,439","4,799","1,080",22.5
2003,"144,132","14,988","2,632",17.6
2004,"120,748","27,387","3,839",14
2005,"92,370","44,119","6,680",15.1
2006,"65,467","65,619","8,714",13.3
2007,"38,963","103,077","11,504",11.2


In [3]:
print(f"Total reserve {m.reserve.sum():,.0f}, Mack s.e. {m.total_se:,.0f} (CV {m.total_se / m.reserve.sum():.1%})")

Total reserve 262,475, Mack s.e. 19,007 (CV 7.2%)


## 2. One-year view: Merz-Wüthrich (2008)
Uncertainty of the claims development result over the next calendar year only, which is the horizon of Solvency II and the SST.
The implementation reproduces the published Merz-Wüthrich examples (see `tests/test_merz_wuthrich.py`).

In [4]:
mw = merz_wuthrich(paid)
pd.DataFrame({"reserve": mw.reserve, "one_year_cdr_se": mw.cdr_se, "mack_ultimate_se": mw.mack_se}, index=seg.origins).round(0)

,reserve,one_year_cdr_se,mack_ultimate_se
1998,0,0,0
1999,100,70,70
2000,538,144,160
2001,"1,848",294,337
2002,"4,799","1,031","1,080"
2003,"14,988","2,346","2,632"
2004,"27,387","2,963","3,839"
2005,"44,119","5,642","6,680"
2006,"65,619","5,872","8,714"
2007,"103,077","7,142","11,504"


In [5]:
print(f"Total: one-year s.e. {mw.cdr_total_se:,.0f} vs ultimate s.e. {mw.mack_total_se:,.0f} "
      f"-> ratio {mw.one_year_share:.2f}")

Total: one-year s.e. 13,738 vs ultimate s.e. 19,007 -> ratio 0.72


Cross-check with a re-reserving ("actuary-in-the-box") ODP bootstrap: simulate next year's diagonal, re-run the chain ladder, record the CDR.

In [6]:
one = odp_one_year_cdr(paid, n_sims=10_000, seed=2026)
ult = odp_bootstrap(paid, n_sims=10_000, seed=2026)
loss = -one.total
print(f"bootstrap one-year s.d. {loss.std():,.0f}, mean CDR {one.total.mean():,.0f}")
print(f"one-year 99.5% VaR of the loss: {np.quantile(loss, 0.995) - loss.mean():,.0f}")
print(f"ultimate 99.5% quantile minus mean: {np.quantile(ult.total, 0.995) - ult.total.mean():,.0f}")

bootstrap one-year s.d. 13,242, mean CDR -260
one-year 99.5% VaR of the loss: 36,681
ultimate 99.5% quantile minus mean: 54,561


## 3. Rate-adjusted (on-levelled) Bornhuetter-Ferguson prior
The original prior is the average loss ratio of AY 1998-2002 (78%). Loss ratios fell steadily afterwards (rate increases).
We estimate the trend from data known at 31.12.2007 only: weighted regression of the log incurred CL loss ratio on accident year,
weights = used-up premium P / CDF.

In [7]:
cl_i = chain_ladder(inc, tail=1.0)
trend = loss_ratio_trend(cl_i.ultimate, seg.premium, cl_i.cdf, seg.origins)
print(f"annual loss-ratio trend {trend.annual_trend:+.2%} (s.e. of slope {trend.slope_se:.3f}, t = {trend.t_stat:.1f})")
elr_flat = apriori_elr_from_mature_years(cl_i.ultimate, seg.premium, 5)
elr_trend = trended_mature_prior(cl_i.ultimate, seg.premium, trend.annual_trend, 5)
_, elr_cc = generalised_cape_cod(cl_i.latest, cl_i.cdf, seg.premium, trend.annual_trend)
pd.DataFrame({"cl_lr": cl_i.ultimate / seg.premium, "prior_flat": elr_flat, "prior_trended": elr_trend,
              "cape_cod_trended": elr_cc, "actual_lr_120m (hold-out)": seg.incurred[:, -1] / seg.premium},
             index=seg.origins).mul(100).round(1)  # in %

annual loss-ratio trend -4.54% (s.e. of slope 0.011, t = -4.2)


,cl_lr,prior_flat,prior_trended,cape_cod_trended,actual_lr_120m (hold-out)
1998,74.8,78.3,87.2,86.6,74.8
1999,74.2,78.3,83.3,82.6,74.1
2000,84.2,78.3,79.5,78.9,84.4
2001,87,78.3,75.9,75.3,87
2002,71.9,78.3,72.4,71.9,72.3
2003,75.4,78.3,69.1,68.6,74.7
2004,64.6,78.3,66,65.5,64.2
2005,58.4,78.3,63,62.5,57.4
2006,56.5,78.3,60.1,59.7,56.9
2007,56.9,78.3,57.4,57,52.2


Run-off check for the three most recent accident years (where BF drives the selection): incurred emergence 2008-2016 to age 120.

In [8]:
recent = slice(7, 10)
actual = (seg.incurred[recent, -1] - cl_i.latest[recent]).sum()
rows = {}
for name, elr in (("BF, flat prior", elr_flat), ("BF, on-levelled prior", elr_trend)):
    pred = (bornhuetter_ferguson(cl_i.latest, cl_i.cdf, seg.premium, elr) - cl_i.latest)[recent].sum()
    rows[name] = {"predicted": pred, "actual": actual, "actual_vs_predicted_%": 100 * (actual / pred - 1)}
rows["Chain ladder"] = {"predicted": cl_i.reserve[recent].sum(), "actual": actual,
                        "actual_vs_predicted_%": 100 * (actual / cl_i.reserve[recent].sum() - 1)}
pd.DataFrame(rows).T.round(1)

,predicted,actual,actual_vs_predicted_%
"BF, flat prior","195,603","130,267",-33.4
"BF, on-levelled prior","148,150","130,267",-12.1
Chain ladder,"142,498","130,267",-8.6


## 4. Tail-factor sensitivity
Five defensible paid tails; the lognormal spread used in the tail-randomised bootstrap spans them.

In [9]:
cand = paid_tail_candidates(paid, inc)
total_ult = m.ultimate.sum()
pd.DataFrame({"tail": {k: round(t, 4) for k, t in cand.items()},
              "unpaid": {k: round(t * total_ult - m.latest.sum()) for k, t in cand.items()}})

,tail,unpaid
none,1,262475
paid_exponential,1.001,264000
paid_inverse_power,1.005,268936
inc_to_paid_x_inc_exponential,1.006,269975
inc_to_paid_x_inc_inverse_power,1.009,273355


In [10]:
print(f"log-sd of (tail - 1) in the randomised bootstrap: {lognormal_tail_sigma(cand):.2f}")

log-sd of (tail - 1) in the randomised bootstrap: 0.60


## Takeaways for this segment
* One-year uncertainty is about 70% of the ultimate one; that ratio is what links a run-off reserve-risk view to SST / Solvency II.
* The on-levelled prior removes most of the BF overshoot that the flat 78% prior produced.
* For commercial auto the tail is short, so tail choice moves the reserve by only a few percent (unlike workers' comp, see README).